[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rewrite/rewrite/examples/mechanics/stiffness.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/EelcoHoogendoorn/numga?ref=rewrite)

# Planar Rigid-Body Stiffness & Vibration Modes in PGA2D

When a rigid body is suspended by elastic springs, our goal is to compute restoring forces and find natural vibration frequencies and mode shapes.

Rather than assembling coordinate stiffness matrices and moment arms by hand, this notebook uses **extensors** (linear maps between blade subspaces):
spring lines directly measure displacement into extension, Hooke's law compiles into a rank-1 stiffness extensor, and normal vibration modes emerge from a single generalized eigensolve between stiffness and inertia.


In [ ]:
# Setup environment: clone repository and configure paths if running in Colab
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import os, shutil, subprocess
    os.chdir("/content")
    repo_dir = Path("/content/numga_repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", "rewrite", "https://github.com/EelcoHoogendoorn/numga.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", "rewrite"], check=False)
    rewrite_dir = repo_dir / "rewrite"
    src_dir = rewrite_dir / "src"
    for p in [str(src_dir), str(rewrite_dir)]:
        if p not in sys.path:
            sys.path.insert(0, p)

# Ensure rewrite root is in sys.path when running locally:
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "examples" / "mechanics").is_dir():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
    if (cand / "rewrite" / "examples" / "mechanics").is_dir():
        p = str(cand / "rewrite")
        if p not in sys.path:
            sys.path.insert(0, p)
        break


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

from numga import NumpyContext
from numga.algebras import PGA2D

# Bind 2D Projective Geometric Algebra (PGA2D):
ctx = NumpyContext(PGA2D)
mv = ctx.multivector

# Blade Subspaces:
Scalar = PGA2D.gatype.scalar()              # Grade 0: scalar real values
Point = PGA2D.gatype.antivector()           # Grade 2: projective points in the 2D plane
Twist = PGA2D.gatype.bivector()             # Grade 2 dual / Lie algebra: infinitesimal rigid motions & velocities
Wrench = PGA2D.gatype.vector()              # Grade 1: lines of action for forces, torques, and springs

# Extensors (linear maps between blade subspaces):
SpringExtension = PGA2D.gatype((Scalar, Twist))   # Scalar <- Twist: measures spring extension from body motion
Stiffness = PGA2D.gatype((Wrench, Twist))         # Wrench <- Twist: restoring force and torque from body motion
Inertia = PGA2D.gatype((Wrench, Twist))           # Wrench <- Twist: kinetic momentum wrench from twist velocity

print("PGA2D Algebra and Extensor types initialized successfully.")


## 1. Rigid Body Geometry & Spring Lines

A uniform 2x1 rectangular plate of mass 1 is suspended in the plane by elastic springs attached to fixed anchors.

In PGA, a spring is simply the line joining its anchor to its attachment point: `lines = (anchors & attachments).normalized()`. This normalized line represents both the spring's line of action and a physical measurement tool.


In [ ]:
from examples.mechanics.stiffness import suspension

# Load both suspension systems (two vertical springs vs. two vertical + one angled spring):
system_parallel = suspension(angled_spring=False)
system_angled = suspension(angled_spring=True)

# Compute spring lines of action in PGA (joining fixed anchor to body attachment point):
spring_lines = (system_parallel.anchors & system_parallel.attachments).normalized()

print(f"Parallel suspension: {system_parallel.attachments.shape[0]} springs, total mass = {system_parallel.masses.kernel.sum():.1f}")
print(f"Angled suspension  : {system_angled.attachments.shape[0]} springs, total mass = {system_angled.masses.kernel.sum():.1f}")
print("Spring line GAType :", spring_lines.gatype)


## 2. Building the Stiffness Extensor from Springs

Hooke's law states that restoring force is proportional to extension along the spring's line of action.

In PGA, pairing a spring line with an open `Twist` measures extension: `extension = Twist & lines`. Multiplying by the line and spring constant compiles directly into the stiffness extensor: `stiffness = (lines * extension * k).sum()`.


In [ ]:
# 1. Measure spring extension from an open rigid-body motion (linear form: Twist -> Scalar):
extension = Twist & spring_lines

# 2. Compile Hooke's law into a stiffness extensor mapping body displacement to opposing wrench:
stiffness_parallel = (spring_lines * extension * system_parallel.stiffnesses).sum(axis=0)

print("Stiffness extensor GAType:", stiffness_parallel.gatype)
print("Kernel shape             :", stiffness_parallel.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Stiffness kernel matrix (two vertical springs):\n", np.round(stiffness_parallel.kernel, 3))


## 3. Building the Inertia Extensor from Mass Points

The inertia extensor maps twist velocity to kinetic momentum wrench (`Wrench <- Twist`).

Each mass point contributes a rate-to-momentum extensor: `p & (p.commutator(Twist)) * m`. Summing across mass points yields the physical inertia extensor, integrating both total mass and polar moment of inertia.


In [ ]:
# Inertia extensor mapping twist velocity to momentum wrench:
# (mass_points & mass_points.commutator(Twist) * masses).sum(axis=0)
inertia = (system_parallel.mass_points & system_parallel.mass_points.commutator(Twist) * system_parallel.masses).sum(axis=0)

print("Inertia extensor GAType:", inertia.gatype)
print("Kernel shape           :", inertia.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Inertia kernel matrix:\n", np.round(inertia.kernel, 3))


## 4. Modal Analysis via Generalized Bilinear Eigensolve

Natural vibration modes solve the generalized eigenvalue problem between elastic potential energy and kinetic energy.

In `numga`, `(Twist & stiffness).eigh(Twist & inertia)` solves this directly on the two bilinear forms without matrix inversion. The eigenvalues give natural frequencies, while Lie commutators give physical mode displacements.


In [ ]:
from examples.mechanics.stiffness import mode_case

def solve_modes(system, angled: bool):
    """Assemble stiffness and inertia extensors and solve for normal modes."""
    # 1. Measure spring lines and compile stiffness extensor:
    lines: Wrench = (system.anchors & system.attachments).normalized()
    extension: SpringExtension = Twist & lines
    stiffness: Stiffness = (lines * extension * system.stiffnesses).sum(axis=0)

    # 2. Compile inertia extensor from mass points:
    inertia: Inertia = (system.mass_points & system.mass_points.commutator(Twist) * system.masses).sum(axis=0)

    # 3. Solve generalized eigenvalue problem directly on bilinear energy forms:
    #    (Twist & stiffness) is the elastic potential form; (Twist & inertia) is the kinetic energy form:
    values, modes = (Twist & stiffness).eigh(Twist & inertia)

    # 4. Extract physical mode displacements across the body using Lie algebra commutators:
    body_offsets = system.body[None, :].commutator(modes[:, None])
    attachment_offsets = system.attachments[None, :].commutator(modes[:, None])
    extensions = extension(modes[:, None])

    return mode_case(system, values, body_offsets, attachment_offsets, extensions, angled)


# Solve normal modes for both suspensions:
case_parallel = solve_modes(system_parallel, angled=False)
case_angled = solve_modes(system_angled, angled=True)
cases = [case_parallel, case_angled]

print("=== Case A: Two Vertical Springs ===")
for label, freq in zip(case_parallel.labels, case_parallel.frequencies):
    detail = "0.00 Hz (free mechanism)" if freq == 0 else f"{freq:.3f} Hz"
    print(f"  {label:<16}: {detail}")

print("\n=== Case B: Two Vertical + One Angled Spring ===")
for label, freq in zip(case_angled.labels, case_angled.frequencies):
    print(f"  {label:<16}: {freq:.3f} Hz")


## 5. Visualizing Natural Vibration Modes

Each row shows the three independent small-motion modes of the rigid body.

Displacements are amplified for clarity. Coiled springs are color-coded: orange for lengthening, blue for shortening, and grey for unchanged. In the top row, sideways motion leaves both vertical springs unchanged to first order (a 0 Hz free slide).


In [ ]:
from examples.mechanics.stiffness_plumbing import draw_modes

# Generate 6-panel mode comparison figure:
fig = draw_modes(cases, plot_path="examples/plots/stiffness.png")
plt.show()


## 6. Harmonic Oscillation Animation

Releasing the plate from rest in each mode demonstrates physical time evolution.

Zero-frequency modes remain statically displaced, while non-zero modes oscillate at their computed natural frequencies. All panels share synchronized physical time.


In [ ]:
from examples.mechanics.stiffness_plumbing import save_animation

# Render and display multi-mode synchronized vibration animation:
gif_path = Path("examples/plots/stiffness.gif")
save_animation(cases, str(gif_path))

if gif_path.exists():
    display(Image(filename=str(gif_path)))


## 7. Summary & Conceptual Synthesis

Having walked through the working pipeline, we can step back and examine how **extensors** structure rigid-body mechanics:

* **Measuring Deformation with Lines (Spring Extension as a Linear Form)**:
  A spring is a line joining an anchor to an attachment point (`lines = anchors & attachments`). Pairing that line with an open twist (`Twist & lines`) directly yields a linear form (`Twist -> Scalar`) measuring spring stretch under any rigid body motion without coordinate projection formulas.

* **Hooke's Law as Rank-1 Extensor Dyads (Stiffness Extensor)**:
  Multiplying the line of action by its extension form and spring constant (`lines * (Twist & lines) * k`) forms a rank-1 extensor dyad mapping body displacements to restoring force and torque (`Wrench <- Twist`). Summing across springs yields the complete stiffness extensor.

* **Kinetic Mass Distribution (Inertia Extensor)**:
  Each mass point contributes a rate-to-momentum extensor: `p & (p.commutator(Twist)) * m`. Summing across mass points integrates total mass, center of mass, and rotational inertia into a single geometric extensor (`Wrench <- Twist`).

* **Solving Modes Directly on Bilinear Energy Forms (Generalized Eigensolve)**:
  Rather than building coordinate mass and stiffness matrices, the vibration eigenvalue problem is solved directly between the potential energy form and kinetic energy form: `(Twist & stiffness).eigh(Twist & inertia)`.

* **Evaluating Physical Displacements (Lie Algebra Commutators)**:
  The motion of any point on the rigid body under an eigenvector twist step is evaluated directly via the Lie algebra commutator (`body.commutator(modes)`), cleanly connecting abstract Lie algebra coordinates to physical geometry.
